# FatFormer-XLA: Notebook Nghiệm Thu Thực Nghiệm GPU Toàn Trình (Phase 7)
## Vận Hành Khép Kín Tasks 7.3 → 7.4 → 7.5 → 7.6 & Phán Quyết Cổng G7.1

> **Nhánh Git**: `hoang` | **Trưởng nhóm Đánh giá (Lead C)**: Lê Việt Hoàng  
> **Mục tiêu**: Thực thi toàn trình trên GPU T4/A100 với dữ liệu ảnh thật CommunityForensics-Eval (2.000 ảnh) và trọng số chuẩn để chốt số liệu đo thật (`evidence_level="measured"`), chính thức nghiệm thu Cổng G7.1 trước khi sang Task 7.11.

### 1. Kết Nối Google Drive & Thiết Lập Không Gian Làm Việc

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Đường dẫn thư mục kho lưu trữ trên Colab
REPO_PATH = '/content/fatformer-xla'
if not os.path.exists(REPO_PATH):
    !git clone -b hoang https://github.com/LeeVHoangtk3/fatformer-xla.git {REPO_PATH}

%cd {REPO_PATH}
!git fetch origin hoang
!git checkout hoang
!git pull origin hoang

# Khóa commit đã resolve tại thời điểm bắt đầu; artifact sẽ ghi lại đúng SHA này.
import subprocess
PINNED_COMMIT_SHA = subprocess.check_output(['git', 'rev-parse', 'origin/hoang'], text=True).strip()
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == PINNED_COMMIT_SHA, 'HEAD không khớp origin/hoang!'
assert subprocess.check_output(['git', 'status', '--porcelain'], text=True).strip() == '', 'Source tree không sạch!'
print('Pinned source commit:', PINNED_COMMIT_SHA)

!git log -1 --oneline
!git status

### 2. Cài Đặt Gói Thư Viện Nền Tảng

In [ ]:
!pip install -q pyyaml ftfy regex scikit-learn matplotlib tqdm pillow

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print(f"VRAM Capacity  : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    raise SystemError("Vui lòng bật GPU Runtime trong Colab (Runtime -> Change runtime type -> T4 GPU)!")

### 3. Đồng Bộ Dữ Liệu CFE & Checkpoint Từ Google Drive Vào Workspace

In [ ]:
# Thư mục trung tâm lưu trữ trên Drive (5TB)
DRIVE_HUB = '/content/drive/MyDrive/FatFormer_Hub'

# 1. Khởi tạo thư mục nội bộ
!mkdir -p DATASET/pretrained DATASET/checkpoints/anchored_recovery_phase_6r DATASET/logs/phase7/calibration

# 2. Liên kết / sao chép backbone CLIP ViT-L-14.pt
if os.path.exists(f'{DRIVE_HUB}/pretrained/ViT-L-14.pt'):
    !cp -n '{DRIVE_HUB}/pretrained/ViT-L-14.pt' DATASET/pretrained/
elif not os.path.exists('DATASET/pretrained/ViT-L-14.pt'):
    # Tải trực tiếp từ nguồn OpenAI nếu trên Drive chưa có
    !wget -c -q -O DATASET/pretrained/ViT-L-14.pt https://openaipublic.azureedge.net/clip/models/b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836/ViT-L-14.pt

# 3. Sao chép 2 checkpoint nguồn
if os.path.exists(f'{DRIVE_HUB}/checkpoints/fatformer_4class_ckpt.pth'):
    !cp -n '{DRIVE_HUB}/checkpoints/fatformer_4class_ckpt.pth' DATASET/pretrained/
if os.path.exists(f'{DRIVE_HUB}/checkpoints/fatformer_anchored_recovery_best.pth'):
    !cp -n '{DRIVE_HUB}/checkpoints/fatformer_anchored_recovery_best.pth' DATASET/checkpoints/anchored_recovery_phase_6r/

# 4. Giải nén tập dữ liệu CommunityForensics-Eval (nếu là file .tar hoặc thư mục)
if os.path.exists(f'{DRIVE_HUB}/datasets/test_communityforensics.tar'):
    !tar -xf '{DRIVE_HUB}/datasets/test_communityforensics.tar' -C DATASET/
elif os.path.exists(f'{DRIVE_HUB}/datasets/test_communityforensics'):
    !ln -s '{DRIVE_HUB}/datasets/test_communityforensics' DATASET/test_communityforensics

# Kiểm tra nhanh cấu trúc tệp
!ls -lh DATASET/pretrained/
!ls -lh DATASET/checkpoints/anchored_recovery_phase_6r/
!ls -lh DATASET/test_communityforensics/ || echo 'Vui lòng kiểm tra đường dẫn CFE dataset!'

### 4. Thực Thi Trình Điều Phối Toàn Trình GPU Acceptance (Tasks 7.3 → 7.6 & G7.1)

In [ ]:
# Chạy trình điều phối GPU runner tự động hoàn toàn (Realtime Streaming & Fail-Loud)
import subprocess
import sys

cmd = [sys.executable, "-u", "tools/run_phase7_acceptance.py", "--config", "configs/phase7_gpu_acceptance.yaml"]
print(f">>> Đang khởi chạy: {' '.join(cmd)}\n")

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True
)

for line in process.stdout:
    print(line, end="", flush=True)

process.wait()

if process.returncode != 0:
    raise RuntimeError(
        f"[GATE REJECTED] GPU Acceptance Runner kết thúc với lỗi (exit code: {process.returncode})! "
        "Tiến trình dừng ngay lập tức để ngăn ngừa đóng gói artifact không đạt chuẩn."
    )
print("\n[✓] GPU Acceptance Runner hoàn thành thành công và toàn bộ tiêu chí Must của Cổng G7.1 đã được thẩm định!")

### 5. Kiểm Tra Kết Quả Ma Trận & Phán Quyết Cổng G7.1

In [ ]:
import json
matrix_json = 'DATASET/logs/phase7/cfe_v3_baseline_matrix.json'
if os.path.exists(matrix_json):
    with open(matrix_json, 'r') as f:
        data = json.load(f)
    print("Evidence Level :", data.get('evidence_level'))
    print("Gate Verdict   :", data.get('gate_verdict'))
    print("Must Pass      :", data.get('pass_must'))
    print("Stretch Pass   :", data.get('pass_stretch'))
    
    # Hiển thị bảng Markdown tóm tắt
    from IPython.display import display, Markdown
    with open('DATASET/logs/phase7/cfe_v3_baseline_matrix.md', 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))

### 6. Đóng Gói Toàn Bộ Artifact Đo Thật Xuất Về Google Drive / Local

In [ ]:
import os
# Kiểm tra tính toàn vẹn của artifact trước khi đóng gói
assert os.path.exists('DATASET/logs/phase7/cfe_v3_baseline_matrix.json'), "Thiếu cfe_v3_baseline_matrix.json!"
assert os.path.exists('DATASET/checkpoints/phase7/fatformer_wise_ft_v2_best_metadata.json'), "Thiếu fatformer_wise_ft_v2_best_metadata.json!"
assert os.path.exists('DATASET/checkpoints/phase7/fatformer_wise_ft_v2_best.pth'), "Thiếu fatformer_wise_ft_v2_best.pth!"

# Nén bằng Python và fail-loud nếu bất kỳ artifact bắt buộc nào không thể đọc.
import zipfile
archive_path = 'phase7_measured_artifacts.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for root, _, names in os.walk('DATASET/logs/phase7'):
        for name in names:
            path = os.path.join(root, name)
            archive.write(path, arcname=path)
    for path in [
        'DATASET/checkpoints/phase7/fatformer_wise_ft_v2_best.pth',
        'DATASET/checkpoints/phase7/fatformer_wise_ft_v2_best_metadata.json',
    ]:
        archive.write(path, arcname=path)
assert os.path.getsize(archive_path) > 0, 'Artifact ZIP rỗng!'
if os.path.exists(DRIVE_HUB):
    !cp phase7_measured_artifacts.zip '{DRIVE_HUB}/phase7_measured_artifacts.zip'
    print(f"[✓] Đã sao lưu artifact đo thật lên Google Drive: {DRIVE_HUB}/phase7_measured_artifacts.zip")

from google.colab import files
files.download('phase7_measured_artifacts.zip')